In [ ]:
import os
import sys
import pathlib
import pandas as pd
import numpy as np
import geopandas as gpd
import matsim
import importlib
from tqdm import tqdm

# Config

In [ ]:
PROJECT_ROOT = pathlib.Path.cwd()
GENERAL_TOOL_PATH = PROJECT_ROOT / "python" / "tools"

SCENARIOS_DIR = PROJECT_ROOT / "output" / "collabReceiverDistantCarrier" / "grid20x20"
# Add the general tool path to the system path
if str(GENERAL_TOOL_PATH) not in sys.path:
    sys.path.append(str(GENERAL_TOOL_PATH))


In [ ]:
import matsim_freight_output_reader
importlib.reload(matsim_freight_output_reader)

# Utils

In [ ]:
def derive_scenario_param_from_name(scenario_name):
    """
    Derives the scenario parameter from the scenario name.
    """
    split_name = scenario_name.split('-')
    depot_loc = split_name[0][:4]
    receiver_distribution = split_name[1]
    allocation_factor = split_name[-4][2:]
    collab_cost = split_name[-3][1:]
    allocation_method = split_name[-2]
    instance_number = split_name[-1][1:]
    return (depot_loc, receiver_distribution, allocation_factor, collab_cost, allocation_method, instance_number)


In [ ]:
def load_agg_results(
    scenarios_dir,
    origin_tw=(6, 7),
    carrier_agg_cols=[
        "carrier_id",
        "selected_plan_score",
        "selected_plan_attribute_jspritScore",
        "vkt_km",
        "ton_km_travelled",
        "cnpi",
        "transit_time_h",
        "travelled_time_h",
        "transit_vkt_km",
        "non_transit_vkt_km",
    ],
    receiver_agg_cols=[
        "receiver_id",
        "linkId",
        "selected_plan_score",
    ],
):
    scenarios = matsim_freight_output_reader.discover_output_scenarios(scenarios_dir)
    agg_carrier_df = pd.DataFrame()
    agg_receiver_df = pd.DataFrame()

    """ For-loop to load and process all the results from each scenario """
    for scenario in tqdm(scenarios.itertuples(), total=len(scenarios), desc="Loading scenarios"):
        scenario_results = matsim_freight_output_reader.read_scenario_output(
            scenario.scenario_path
        )
        scenario_params = derive_scenario_param_from_name(scenario.scenario_name)

        carrier_results = scenario_results["carriers"]["carriers"]
        carrier_results = carrier_results[carrier_agg_cols]
        iter0_carrier_results = scenario_results["iter0_carriers"]["carriers"]
        iter0_carrier_results = iter0_carrier_results[carrier_agg_cols]
        last_iter_receiver_results = scenario_results["last_iter_receivers"]["receivers"]
        last_iter_receiver_results = last_iter_receiver_results[receiver_agg_cols]
        # Rename all the columns to indicate that they are from iter0
        iter0_carrier_results = iter0_carrier_results.rename(
            columns={col: f"iter0_{col}" for col in iter0_carrier_results.columns if col != "carrier_id"}
        )
        collab_results = scenario_results["collaboration"]["allocations"]
        collab_results = collab_results.drop(columns=['allocation_index'])
        # merge iter0 and last iter results with the carrier results
        carrier_results = pd.merge(
            carrier_results,
            iter0_carrier_results,
            how="left",
            left_on="carrier_id",
            right_on="carrier_id",
        )
        # Merge the allocation results with the receiver and carrier results
        carrier_results = pd.merge(
            carrier_results,
            collab_results,
            how="left",
            left_on="carrier_id",
            right_on="collaboratorId",
        )
        receiver_results = pd.merge(
            last_iter_receiver_results,
            collab_results,
            how="left",
            left_on="receiver_id",
            right_on="collaboratorId",
        )

        # Add the scenario parameters to the results
        for param_name, param_value in zip(
            [
                "depot_loc",
                "receiver_distribution",
                "allocation_factor",
                "collab_cost",
                "allocation_method",
                "instance_number",
            ],
            scenario_params,
        ):
            carrier_results[param_name] = param_value
            receiver_results[param_name] = param_value
        
        agg_carrier_df = pd.concat([agg_carrier_df, carrier_results], ignore_index=True)
        agg_receiver_df = pd.concat([agg_receiver_df, receiver_results], ignore_index=True)
    return agg_carrier_df, agg_receiver_df

In [ ]:
test_agg_carrier_results, test_agg_receiver_results = load_agg_results(SCENARIOS_DIR)

In [ ]:
test_agg_carrier_results

In [ ]:
test_agg_carrier_results['carrier_cost_savings'] = test_agg_carrier_results['selected_plan_score'] - test_agg_carrier_results['iter0_selected_plan_score']
test_agg_carrier_results['carrier_vrp_score_savings'] = test_agg_carrier_results['selected_plan_attribute_jspritScore'] - test_agg_carrier_results['iter0_selected_plan_attribute_jspritScore']

In [ ]:
test_agg_receiver_results

# Test

## Test on agg dfs

In [ ]:
def plot_box_across_scenarios(
    agg_df, group_cols=None, value_col='cnpi', show_points=True
):
    """Plot the distribution of a metric for every scenario.

    A scenario is defined by the unique combination of ``group_cols``. Rows
    with the same combination (for example, different ``instance_number``
    values) form the distribution shown by one box.

    Parameters
    ----------
    agg_df : pandas.DataFrame
        Aggregated carrier or receiver results.
    group_cols : sequence of str or str, optional
        Columns that define a scenario. If omitted, the standard scenario
        parameter columns are used. ``instance_number`` should normally not
        be included because instances are the observations inside each box.
    value_col : str, default ``'cnpi'``
        Numeric metric to plot.
    show_points : bool, default ``True``
        Overlay the individual observations (normally scenario instances).

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
        The created plot, so callers can further customise or save it.

    Examples
    --------
    >>> fig, ax = plot_box_across_scenarios(
    ...     test_agg_carrier_results, value_col='cnpi'
    ... )
    """
    import matplotlib.pyplot as plt

    if group_cols is None:
        group_cols = [
            'depot_loc',
            'receiver_distribution',
            'allocation_factor',
            'collab_cost',
            'allocation_method',
        ]
    elif isinstance(group_cols, str):
        group_cols = [group_cols]
    else:
        group_cols = list(group_cols)

    if not group_cols:
        raise ValueError('group_cols must contain at least one column')

    required_cols = group_cols + [value_col]
    missing_cols = [col for col in required_cols if col not in agg_df.columns]
    if missing_cols:
        raise KeyError(f'Columns not found in agg_df: {missing_cols}')

    plot_df = agg_df[required_cols].copy()
    plot_df[value_col] = pd.to_numeric(plot_df[value_col], errors='coerce')
    plot_df = plot_df.dropna(subset=[value_col])
    if plot_df.empty:
        raise ValueError(f'{value_col!r} contains no numeric, non-missing values')

    grouped = plot_df.groupby(group_cols, sort=False, dropna=False)[value_col]
    varying_indices = [
        index
        for index, col in enumerate(group_cols)
        if plot_df[col].nunique(dropna=False) > 1
    ]
    label_indices = varying_indices or list(range(len(group_cols)))
    label_cols = [group_cols[index] for index in label_indices]
    box_values = []
    scenario_labels = []
    for scenario, values in grouped:
        scenario = scenario if isinstance(scenario, tuple) else (scenario,)
        box_values.append(values.to_numpy())
        scenario_labels.append(
            ' | '.join(str(scenario[index]) for index in label_indices)
        )

    # Restrained colours, fine strokes and compact typography inspired by
    # the visual language commonly used in Nature research figures.
    nature_blue = '#3C5488'
    nature_red = '#E64B35'
    text_color = '#222222'
    grid_color = '#D9D9D9'

    n_scenarios = len(box_values)
    fig_width = max(6.5, min(18, 0.38 * n_scenarios))
    fig_height = 5.0 if n_scenarios <= 12 else 6.2
    rc = {
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'font.size': 8,
        'axes.labelsize': 9,
        'axes.titlesize': 10,
        'xtick.labelsize': 7,
        'ytick.labelsize': 8,
        'axes.linewidth': 0.8,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=150)
        boxplot = ax.boxplot(
            box_values,
            widths=0.56,
            patch_artist=True,
            showfliers=False,
            boxprops={
                'facecolor': '#DCE6F1',
                'edgecolor': nature_blue,
                'linewidth': 1.0,
            },
            whiskerprops={'color': nature_blue, 'linewidth': 0.9},
            capprops={'color': nature_blue, 'linewidth': 0.9},
            medianprops={'color': nature_red, 'linewidth': 1.4},
        )

        if show_points:
            # A fixed seed keeps the jitter stable when the cell is rerun.
            rng = np.random.default_rng(42)
            total_points = sum(len(values) for values in box_values)
            point_size = 12 if total_points <= 500 else 7
            for position, values in enumerate(box_values, start=1):
                jitter = rng.uniform(-0.17, 0.17, size=len(values))
                ax.scatter(
                    position + jitter,
                    values,
                    s=point_size,
                    facecolor=nature_blue,
                    edgecolor='white',
                    linewidth=0.25,
                    alpha=0.62,
                    zorder=3,
                    rasterized=total_points > 1000,
                )

        positions = np.arange(1, n_scenarios + 1)
        label_rotation = 0 if n_scenarios <= 6 else 90
        if n_scenarios <= 6:
            scenario_labels = [label.replace(' | ', '\n') for label in scenario_labels]
        ax.set_xticks(positions)
        ax.set_xticklabels(
            scenario_labels,
            rotation=label_rotation,
            ha='center',
            color=text_color,
        )
        ax.set_xlabel('Scenario: ' + ' | '.join(label_cols), labelpad=8)
        ax.set_ylabel(value_col.replace('_', ' ').upper(), labelpad=7)
        ax.set_title(
            f'{value_col.replace("_", " ").upper()} across scenarios',
            loc='left',
            pad=10,
            fontweight='bold',
            color=text_color,
        )

        ax.set_axisbelow(True)
        ax.yaxis.grid(True, color=grid_color, linewidth=0.55, linestyle='-')
        ax.xaxis.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color(text_color)
        ax.spines['bottom'].set_color(text_color)
        ax.tick_params(axis='x', length=0, pad=4)
        ax.tick_params(axis='y', direction='out', length=3, width=0.7)
        ax.margins(x=0.01)
        fig.tight_layout(pad=0.8)

    return fig, ax


In [ ]:
use_agg_carrier_df = test_agg_carrier_results.copy()
use_agg_carrier_df = use_agg_carrier_df[use_agg_carrier_df['receiver_distribution'] == 'clustered']
use_agg_carrier_df = use_agg_carrier_df[use_agg_carrier_df['collab_cost'] == '0.0056']
use_agg_carrier_df

In [ ]:
plot_box_across_scenarios(use_agg_carrier_df, 
    group_cols=None, 
    value_col='selected_plan_attribute_jspritScore'
    )

In [ ]:
plot_box_across_scenarios(use_agg_carrier_df, 
    group_cols=None, 
    value_col='selected_plan_score'
    )

In [ ]:
plot_box_across_scenarios(use_agg_carrier_df, 
    group_cols=None, 
    value_col='vkt_km'
    )

In [ ]:
plot_box_across_scenarios(use_agg_carrier_df, 
    group_cols=None, 
    value_col='ton_km_travelled'
    )

In [ ]:
use_agg_receiver_df = test_agg_receiver_results.copy()
use_agg_receiver_df = use_agg_receiver_df[use_agg_receiver_df['receiver_distribution'] == 'clustered']
use_agg_receiver_df = use_agg_receiver_df[use_agg_receiver_df['collab_cost'] == '0.0028']
use_agg_receiver_df = use_agg_receiver_df[use_agg_receiver_df['instance_number'] == '00']
use_agg_receiver_df

In [ ]:
(use_agg_receiver_df[use_agg_receiver_df['receiver_id'] == 'receiver_08']).plot(kind='line', x='depot_loc', y='value')

## Test on func

In [ ]:
matsim_freight_output_reader.discover_output_scenarios(SCENARIOS_DIR)

In [ ]:
scenario = (
    "output/collabReceiverDistantCarrier/grid20x20/"
    "dc02-dispersed-centeredChessboardArea-penSweep-af0.80-p0.0056-exactShapley-i09"
)

result = matsim_freight_output_reader.read_scenario_output(scenario)

chains = result["events"]["travel_chains"]
vehicles = result["events"]["vehicle_summary"]
tours = result["events"]["tours"]
shipments = result["events"]["shipments"]

receivers = result["receivers"]["receivers"]
last_iter_receivers = result["last_iter_receivers"]["receivers"]
carriers = result["carriers"]["carriers"]
iter0_carriers = result["iter0_carriers"]["carriers"]
last_iter_carriers = result["last_iter_carriers"]["carriers"]
allocations = result["collaboration"]["allocations"]
network_links = result["network"]["links"]

In [ ]:
result['carriers'].keys()

In [ ]:
result['carriers']['vehicles']

In [ ]:
chains

In [ ]:
vehicles

In [ ]:
shipments

In [ ]:
shipments.groupby('vehicle_id').agg({'capacity_demand': 'sum', })

In [ ]:
tours

In [ ]:
iter0_carriers

In [ ]:
carriers

In [ ]:
last_iter_carriers

In [ ]:
receivers

In [ ]:
receivers['selected_plan_score'].sum()

In [ ]:
last_iter_receivers

In [ ]:
last_iter_receivers['selected_plan_score'].sum()

In [ ]:
allocations

In [ ]:
network_links

In [ ]:
scenarios = matsim_freight_output_reader.discover_output_scenarios(
    "output/collabReceiverDistantCarrier/grid20x20"
)

# only read the network once, since it is the same for all scenarios
network = matsim_freight_output_reader.read_network(scenarios.iloc[0]["network_path"])

vehicle_results = []

for scenario in scenarios.itertuples():
    events = matsim_freight_output_reader.read_freight_events(
        scenario.events_path,
        network=network,
    )

    summary = events["vehicle_summary"].assign(
        scenario_name=scenario.scenario_name
    )
    vehicle_results.append(summary)

all_vehicle_results = pd.concat(vehicle_results, ignore_index=True)

In [ ]:
all_vehicle_results